# M05a · Dérivées, Taylor et calcul numérique — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m05a_derivees_taylor/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m05a_derivees_taylor/03_notebook.ipynb`, créée par `python tools/start_chapter.py M05a`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M05a.1 à M05a.10, M05a.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M05a.18 | Évaluer un polynôme de Taylor · `taylor_polynomial` | 🔨 | ★ | 20 |
|  | M05a.19 | La méthode de Newton, avec ses cas d'échec · `newton` | 🔨 | ★ | 20 |
|  | M05a.20 | L'extrapolation de Richardson : annuler le terme d'erreur en h² · `richardson_derivative` | 🔨 | ★★ | 20 |
|  | M05a.21 | L'epsilon machine, en float64, float32 et float16 · `machine_epsilon` | 🔨 | ★ | 15 |
|  | M05a.22 | Un log-sum-exp stable · `logsumexp` | 🔨 | ★ | 15 |
|  | M05a.23 | SymPy `series` et `numpy.finfo` contre tes fonctions | 📦 | ★ | 15 |
| B | M05a.24 | Prédire : Newton depuis un mauvais point de départ | 🔮 | ★ | 10 |
|  | M05a.25 | Taylor de plus en plus loin : où le polynôme décroche | 🔬 | ★★ | 20 |
|  | M05a.26 | Une fonction et ses polynômes de Taylor d'ordre 1, 2 et 3 | 🎨 | ★ | 20 |
| C | M05a.27 | Chasse à l'erreur : une softmax qui renvoie `nan` | 🐛 | ★ | 15 |
|  | M05a.28 | Erreur relative ou absolue : choisir la tolérance d'un test | 🛠️ | ★ | 10 |
|  | M05a.29 | Défi : une racine cubique à 10⁻¹² en le moins d'itérations possible | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05a")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : dérivée seconde et convexité, extrema, polynômes de Taylor et reste, méthode de Newton, erreur des différences finies, flottants, meilleur pas, log-sum-exp.
- Écrire les cinq fonctions de `mymath/approx.py` et les tester, puis utiliser SymPy et `numpy.finfo`.
- Prédire Newton depuis un mauvais départ, voir une série de Taylor décrocher, tracer des polynômes de Taylor, chasser les erreurs d'une softmax, comparer des flottants, calculer une racine cubique en quelques pas.

**Rappel express.** $f'' \ge 0$ : convexe ; $P_n(x) = \sum_{k=0}^{n} \frac{f^{(k)}(a)}{k!}(x - a)^k$, reste $\le \frac{M}{(n+1)!}|x - a|^{n+1}$ ; Newton : $x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}$ ; différence avant : erreur $\approx \frac{h}{2}f''$, centrée : $\approx \frac{h^2}{6}f'''$, Richardson : $\frac{4D(h/2) - D(h)}{3}$ ; `float64` : $\varepsilon = 2^{-52}$ ; $\text{LSE}(\mathbf{v}) = m + \ln\sum_i e^{v_i - m}$, $m = \max_i v_i$, entre $m$ et $m + \ln n$.

## Partie 0 · Vérifier tes exercices papier (✏️ M05a.1 à M05a.10, 🔗 M05a.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"7/9"` ; une expression en $x$ entre guillemets, `"2*x**3 - x"` ;
- une liste `[-1.5, 2.5]`, un intervalle entre guillemets `"[-4, 6]"`, `True` / `False`, une lettre entre guillemets, `"B"`, ou un texte entre guillemets, `"10.01"`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 4) is accepted too, as "3/4"

**Ex M05a.1 — Dérivées secondes et convexité**

In [ ]:
# Ex M05a.1: the values you found on paper
answer_M05a_1a = ...  # a) an integer
answer_M05a_1b = ...  # b) an interval, as text
answer_M05a_1c = ...  # c) a list
answer_M05a_1d = ...  # d) a number (4 decimals)
answer_M05a_1e = ...  # e) True or False
answer_M05a_1f = ...  # f) a number (1 decimal)
answer_M05a_1g = ...  # g) an integer
answer_M05a_1h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M05a_1a, answer_M05a_1b, answer_M05a_1c, answer_M05a_1d, answer_M05a_1e, answer_M05a_1f, answer_M05a_1g, answer_M05a_1h]):
    wbm.check(f"M05a.1{letter}", answer)

**Ex M05a.2 — Extrema locaux et globaux sur un intervalle fermé**

In [ ]:
# Ex M05a.2: the values you found on paper
answer_M05a_2a = ...  # a) a list
answer_M05a_2b = ...  # b) a list
answer_M05a_2c = ...  # c) an integer
answer_M05a_2d = ...  # d) an integer
answer_M05a_2e = ...  # e) an integer
answer_M05a_2f = ...  # f) an integer
answer_M05a_2g = ...  # g) a number (4 decimals)
answer_M05a_2h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M05a_2a, answer_M05a_2b, answer_M05a_2c, answer_M05a_2d, answer_M05a_2e, answer_M05a_2f, answer_M05a_2g, answer_M05a_2h]):
    wbm.check(f"M05a.2{letter}", answer)

**Ex M05a.3 — Points d'inflexion et test de la dérivée seconde**

In [ ]:
# Ex M05a.3: the values you found on paper
answer_M05a_3a = ...  # a) a list
answer_M05a_3b = ...  # b) an integer
answer_M05a_3c = ...  # c) an integer
answer_M05a_3d = ...  # d) a letter, as text
answer_M05a_3e = ...  # e) a list
answer_M05a_3f = ...  # f) an integer
answer_M05a_3g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M05a_3a, answer_M05a_3b, answer_M05a_3c, answer_M05a_3d, answer_M05a_3e, answer_M05a_3f, answer_M05a_3g]):
    wbm.check(f"M05a.3{letter}", answer)

**Ex M05a.4 — Polynômes de Taylor de trois fonctions usuelles**

In [ ]:
# Ex M05a.4: the values you found on paper
answer_M05a_4a = ...  # a) an expression in x, as text
answer_M05a_4b = ...  # b) an expression in x, as text
answer_M05a_4c = ...  # c) an expression in x, as text
answer_M05a_4d = ...  # d) a number (4 decimals)
answer_M05a_4e = ...  # e) a number (5 decimals)
answer_M05a_4f = ...  # f) an irreducible fraction, as text
answer_M05a_4g = ...  # g) a number (3 decimals)
answer_M05a_4h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M05a_4a, answer_M05a_4b, answer_M05a_4c, answer_M05a_4d, answer_M05a_4e, answer_M05a_4f, answer_M05a_4g, answer_M05a_4h]):
    wbm.check(f"M05a.4{letter}", answer)

**Ex M05a.5 — Le reste de Taylor : borner l'erreur d'une approximation**

In [ ]:
# Ex M05a.5: the values you found on paper
answer_M05a_5a = ...  # a) a number (6 decimals)
answer_M05a_5b = ...  # b) a number (7 decimals)
answer_M05a_5c = ...  # c) a number (7 decimals)
answer_M05a_5d = ...  # d) an integer
answer_M05a_5e = ...  # e) a number (6 decimals)
answer_M05a_5f = ...  # f) a number (7 decimals)
answer_M05a_5g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M05a_5a, answer_M05a_5b, answer_M05a_5c, answer_M05a_5d, answer_M05a_5e, answer_M05a_5f, answer_M05a_5g]):
    wbm.check(f"M05a.5{letter}", answer)

**Ex M05a.6 — Newton à la main : une racine carrée en trois pas**

In [ ]:
# Ex M05a.6: the values you found on paper
answer_M05a_6a = ...  # a) an irreducible fraction, as text
answer_M05a_6b = ...  # b) an irreducible fraction, as text
answer_M05a_6c = ...  # c) a number (10 decimals)
answer_M05a_6d = ...  # d) a number (2 decimals)
answer_M05a_6e = ...  # e) a number (3 decimals)
answer_M05a_6f = ...  # f) a number (4 decimals)
answer_M05a_6g = ...  # g) True or False
answer_M05a_6h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M05a_6a, answer_M05a_6b, answer_M05a_6c, answer_M05a_6d, answer_M05a_6e, answer_M05a_6f, answer_M05a_6g, answer_M05a_6h]):
    wbm.check(f"M05a.6{letter}", answer)

**Ex M05a.7 — L'erreur d'une différence avant et d'une différence centrée, par Taylor**

In [ ]:
# Ex M05a.7: the values you found on paper
answer_M05a_7a = ...  # a) an irreducible fraction, as text
answer_M05a_7b = ...  # b) a number (5 decimals)
answer_M05a_7c = ...  # c) a number (5 decimals)
answer_M05a_7d = ...  # d) a number (6 decimals)
answer_M05a_7e = ...  # e) a number (6 decimals)
answer_M05a_7f = ...  # f) an irreducible fraction, as text
answer_M05a_7g = ...  # g) an integer
answer_M05a_7h = ...  # h) a number (5 decimals)

for letter, answer in zip("abcdefgh", [answer_M05a_7a, answer_M05a_7b, answer_M05a_7c, answer_M05a_7d, answer_M05a_7e, answer_M05a_7f, answer_M05a_7g, answer_M05a_7h]):
    wbm.check(f"M05a.7{letter}", answer)

**Ex M05a.8 — Les flottants : 0,1 en binaire, epsilon machine, nombres représentables**

In [ ]:
# Ex M05a.8: the values you found on paper
answer_M05a_8a = ...  # a) a text, such as "0.11"
answer_M05a_8b = ...  # b) True or False
answer_M05a_8c = ...  # c) a text of 4 digits
answer_M05a_8d = ...  # d) an integer
answer_M05a_8e = ...  # e) an integer
answer_M05a_8f = ...  # f) an integer
answer_M05a_8g = ...  # g) an irreducible fraction, as text
answer_M05a_8h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M05a_8a, answer_M05a_8b, answer_M05a_8c, answer_M05a_8d, answer_M05a_8e, answer_M05a_8f, answer_M05a_8g, answer_M05a_8h]):
    wbm.check(f"M05a.8{letter}", answer)

**Ex M05a.9 — Le meilleur pas par le calcul : minimiser une borne d'erreur**

In [ ]:
# Ex M05a.9: the values you found on paper
answer_M05a_9a = ...  # a) a number (3 decimals)
answer_M05a_9b = ...  # b) a number (3 decimals)
answer_M05a_9c = ...  # c) a number (3 decimals)
answer_M05a_9d = ...  # d) a number (2 decimals)
answer_M05a_9e = ...  # e) a number (1 decimal)
answer_M05a_9f = ...  # f) a number (2 decimals)
answer_M05a_9g = ...  # g) a number (4 decimals)
answer_M05a_9h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M05a_9a, answer_M05a_9b, answer_M05a_9c, answer_M05a_9d, answer_M05a_9e, answer_M05a_9f, answer_M05a_9g, answer_M05a_9h]):
    wbm.check(f"M05a.9{letter}", answer)

**Ex M05a.10 — log-sum-exp à la main : la softmax stable de trois scores**

In [ ]:
# Ex M05a.10: the values you found on paper
answer_M05a_10a = ...  # a) True or False
answer_M05a_10b = ...  # b) an integer
answer_M05a_10c = ...  # c) a number (4 decimals)
answer_M05a_10d = ...  # d) a list of 3 numbers (4 decimals)
answer_M05a_10e = ...  # e) a number (4 decimals)
answer_M05a_10f = ...  # f) a number (4 decimals)
answer_M05a_10g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_M05a_10a, answer_M05a_10b, answer_M05a_10c, answer_M05a_10d, answer_M05a_10e, answer_M05a_10f, answer_M05a_10g]):
    wbm.check(f"M05a.10{letter}", answer)

**Ex M05a.14 — Pourquoi les bibliothèques de ML calculent log(softmax) d'un seul coup**

In [ ]:
# Ex M05a.14: the values you found on paper
answer_M05a_14a = ...  # a) a number (2 decimals)
answer_M05a_14b = ...  # b) a number (2 decimals)
answer_M05a_14c = ...  # c) True or False
answer_M05a_14d = ...  # d) a number (1 decimal)
answer_M05a_14e = ...  # e) True or False
answer_M05a_14f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M05a_14a, answer_M05a_14b, answer_M05a_14c, answer_M05a_14d, answer_M05a_14e, answer_M05a_14f]):
    wbm.check(f"M05a.14{letter}", answer)

## Partie A · Ta librairie : Taylor, Newton, Richardson, epsilon machine et log-sum-exp

Fiche §3 à §8. Les cinq fonctions de `mymath/approx.py`, puis SymPy `series` et `numpy.finfo`.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args, **kwargs):
    """The name of the exception that function(*args, **kwargs) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳; so do the errors of a code
    that cannot run (a missing import, a misspelt name such as ValueErorr), for the check to name them."""
    try:
        function(*args, **kwargs)
    except (NotImplementedError, NameError, AttributeError, ImportError):
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M05a.18 — Évaluer un polynôme de Taylor · `taylor_polynomial` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la valeur d'un polynôme de Taylor à partir des dérivées en un point, sans dépassement de capacité pour un grand nombre de termes.  
**Prérequis :** fiche §3 · **mymath :** `approx.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/approx.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Le module `math` suffit (`math.exp`, `math.log`, `math.fsum`, `math.inf`, `math.isinf`). `numpy.finfo`, `sys.float_info` et `scipy.special.logsumexp` sont interdits : les tests comparent tes résultats à eux, et un test vérifie que tu ne t'en sers pas. Les nombres renvoyés sont des `float` de Python.

Écris `taylor_polynomial(derivatives, a, x)` (fiche §3) : la valeur en $x$ du polynôme $\sum_{k} \frac{f^{(k)}(a)}{k!}(x - a)^k$, connaissant la liste $[f(a), f'(a), f''(a), \ldots]$ ; une `ValueError` si la liste est vide.

Un piège : pour 200 termes, $k!$ devient un entier de plus de 370 chiffres, que Python ne sait plus convertir en `float` (une `OverflowError`). Garde plutôt à jour le terme $\frac{(x - a)^k}{k!}$ : on passe du terme $k - 1$ au terme $k$ en le multipliant par $\frac{x - a}{k}$.

La vérification calcule :
- a) `taylor_polynomial([0, 1, 0, -1, 0, 1], 0, 0.8)` (6 décimales) ;
- b) `taylor_polynomial([1, 0, -1, 0, 1], 0, 1.0)` (6 décimales) ;
- c) `taylor_polynomial([0, 1, -1, 2], 1, 1.5)` (6 décimales) ;
- d) le nom de l'erreur levée par `taylor_polynomial([], 0, 1.0)` ;
- e) le nom du type de `taylor_polynomial([2, 3], 1, 4)`, en texte ;
- f) `taylor_polynomial([4.0], 10, -3)` (4 décimales) ;
- g) `taylor_polynomial([1.0] * 30, 0, 5.0)` (4 décimales).

Dans ta copie : de quelles fonctions a), b) et c) sont-ils des polynômes de Taylor ? Compare a) à $\sin 0{,}8$, b) à $\cos 1$ et g) à $e^5$. Refais b) à la main.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05a")  # reload your saved file

In [ ]:
with checked("M05a.18a"):
    wbm.check("M05a.18a", mymath.approx.taylor_polynomial([0, 1, 0, -1, 0, 1], 0, 0.8), computed=True)
with checked("M05a.18b"):
    wbm.check("M05a.18b", mymath.approx.taylor_polynomial([1, 0, -1, 0, 1], 0, 1.0), computed=True)
with checked("M05a.18c"):
    wbm.check("M05a.18c", mymath.approx.taylor_polynomial([0, 1, -1, 2], 1, 1.5), computed=True)
with checked("M05a.18d"):
    wbm.check("M05a.18d", raised(mymath.approx.taylor_polynomial, [], 0, 1.0), computed=True)
with checked("M05a.18e"):
    wbm.check("M05a.18e", type(mymath.approx.taylor_polynomial([2, 3], 1, 4)).__name__, computed=True)
with checked("M05a.18f"):
    wbm.check("M05a.18f", mymath.approx.taylor_polynomial([4.0], 10, -3), computed=True)
with checked("M05a.18g"):
    wbm.check("M05a.18g", mymath.approx.taylor_polynomial([1.0] * 30, 0, 5.0), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05a_approx.py", "test_taylor_polynomial_ or test_approx_uses_")

### Ex M05a.19 — La méthode de Newton, avec ses cas d'échec · `newton` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la méthode de Newton avec un critère d'arrêt, et traiter ses échecs : tangente horizontale, pas de convergence.  
**Prérequis :** fiche §4 · **mymath :** `approx.py` · **Parcours :** C, E

Écris `newton(f, fprime, x0, tol=1e-12, max_iter=50)` (fiche §4). L'ordre des tests compte ; à chaque pas :
1. si $f(x)$ vaut exactement 0, renvoie $x$ (c'est déjà un zéro) ;
2. sinon, si $f'(x)$ vaut 0, lève une `ValueError` (la tangente est horizontale) ;
3. sinon, calcule $x_{\text{nouveau}} = x - \frac{f(x)}{f'(x)}$, et renvoie-le si $|x_{\text{nouveau}} - x| < \text{tol}$.

Après `max_iter` pas sans s'arrêter, lève une `RuntimeError`.

La vérification calcule :
- a) `newton(lambda x: x * x - 7, lambda x: 2 * x, 3.0)` (10 décimales) ;
- b) `newton(lambda x: math.exp(-x) - x, lambda x: -math.exp(-x) - 1, 0.0)` (10 décimales) ;
- c) le nom de l'erreur levée par `newton(lambda x: x * x - 2 * x + 5, lambda x: 2 * x - 2, 1.0)` ;
- d) le nom de l'erreur levée par `newton(lambda x: x * x - 7, lambda x: 2 * x, 3.0, max_iter=3)` ;
- e) le nom de l'erreur levée par `newton(lambda x: (x - 2) ** 3, lambda x: 3 * (x - 2) ** 2, 2.0)`, ou `"aucune"` ;
- f) `newton(lambda x: x * x - 7, lambda x: 2 * x, 3.0, tol=0.1)` (7 décimales) ;
- g) le nom du type de `newton(lambda x: x - 5, lambda x: 1, 5)`, en texte.

Dans ta copie : écris les trois premiers pas de a) à la main. Quel nombre b) résout-il ? Pourquoi c) échoue-t-il, et ce polynôme a-t-il des racines réelles ? Pourquoi l'ordre des tests 1 et 2 compte-t-il pour e) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05a")  # reload your saved file

In [ ]:
with checked("M05a.19a"):
    wbm.check("M05a.19a", mymath.approx.newton(lambda x: x * x - 7, lambda x: 2 * x, 3.0), computed=True)
with checked("M05a.19b"):
    wbm.check("M05a.19b", mymath.approx.newton(lambda x: math.exp(-x) - x, lambda x: -math.exp(-x) - 1, 0.0),
              computed=True)
with checked("M05a.19c"):
    wbm.check("M05a.19c", raised(mymath.approx.newton, lambda x: x * x - 2 * x + 5, lambda x: 2 * x - 2, 1.0),
              computed=True)
with checked("M05a.19d"):
    wbm.check("M05a.19d", raised(mymath.approx.newton, lambda x: x * x - 7, lambda x: 2 * x, 3.0, max_iter=3),
              computed=True)
with checked("M05a.19e"):
    wbm.check("M05a.19e", raised(mymath.approx.newton, lambda x: (x - 2) ** 3, lambda x: 3 * (x - 2) ** 2, 2.0),
              computed=True)
with checked("M05a.19f"):
    wbm.check("M05a.19f", mymath.approx.newton(lambda x: x * x - 7, lambda x: 2 * x, 3.0, tol=0.1), computed=True)
with checked("M05a.19g"):
    wbm.check("M05a.19g", type(mymath.approx.newton(lambda x: x - 5, lambda x: 1, 5)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05a_approx.py", "test_newton_")

### Ex M05a.20 — L'extrapolation de Richardson : annuler le terme d'erreur en h² · `richardson_derivative` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer l'extrapolation de Richardson de la différence centrée, et mesurer son ordre.  
**Prérequis :** fiche §5 ; Ex M05a.7 · **mymath :** `approx.py` · **Parcours :** C

Écris `richardson_derivative(f, x, h=0.01)` (fiche §5) : avec la différence centrée $D(h) = \frac{f(x + h) - f(x - h)}{2h}$, renvoie $\frac{4D(h/2) - D(h)}{3}$ ; une `ValueError` si $h \le 0$ (pense à $h = 0$ et à un $h$ négatif).

La vérification calcule :
- a) `richardson_derivative(math.cos, 0.6, 0.1)` (8 décimales) ;
- b) `richardson_derivative(math.log, 3.0)` (10 décimales) ;
- c) le nom de l'erreur levée par `richardson_derivative(math.exp, 1.0, -0.5)` ;
- d) `richardson_derivative(lambda t: t ** 4, 2.0, 0.5)` (4 décimales) ;
- e) le rapport des erreurs de `richardson_derivative(math.exp, 1.0, h)` pour $h = 0{,}2$ et $h = 0{,}1$ (1 décimale) ;
- f) le nom du type de `richardson_derivative(np.sin, 0.5, 0.1)`, en texte ;
- g) l'erreur de a), `richardson_derivative(math.cos, 0.6, 0.1) + math.sin(0.6)`, multipliée par $10^7$ (2 décimales).

Dans ta copie : quelle est la dérivée exacte en a) et en b) ? Calcule la différence centrée $D(0{,}1)$ de a) et son erreur : combien de fois plus grande que g) ? Pourquoi d) est-il exact, alors que $D(0{,}5)$ ne l'est pas (M05a.11, question 7) ? Que dit e) de l'ordre de la méthode ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05a")  # reload your saved file

In [ ]:
with checked("M05a.20a"):
    wbm.check("M05a.20a", mymath.approx.richardson_derivative(math.cos, 0.6, 0.1), computed=True)
with checked("M05a.20b"):
    wbm.check("M05a.20b", mymath.approx.richardson_derivative(math.log, 3.0), computed=True)
with checked("M05a.20c"):
    wbm.check("M05a.20c", raised(mymath.approx.richardson_derivative, math.exp, 1.0, -0.5), computed=True)
with checked("M05a.20d"):
    wbm.check("M05a.20d", mymath.approx.richardson_derivative(lambda t: t ** 4, 2.0, 0.5), computed=True)
with checked("M05a.20e"):
    errors_20 = [abs(mymath.approx.richardson_derivative(math.exp, 1.0, h) - math.e) for h in (0.2, 0.1)]
    wbm.check("M05a.20e", errors_20[0] / errors_20[1], computed=True)
with checked("M05a.20f"):
    wbm.check("M05a.20f", type(mymath.approx.richardson_derivative(np.sin, 0.5, 0.1)).__name__, computed=True)
with checked("M05a.20g"):
    wbm.check("M05a.20g", (mymath.approx.richardson_derivative(math.cos, 0.6, 0.1) + math.sin(0.6)) * 1e7,
              computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05a_approx.py", "test_richardson_derivative_")

### Ex M05a.21 — L'epsilon machine, en float64, float32 et float16 · `machine_epsilon` 🔨 ★ ⏱️ 15 min
**Objectif :** calculer l'epsilon machine d'un type de flottants en le cherchant, puis relier l'écart entre deux flottants à la puissance de 2 du nombre.  
**Prérequis :** fiche §6 ; Ex M05a.8 · **mymath :** `approx.py` · **Parcours :** C

Écris `machine_epsilon(dtype=float)` (fiche §6) : en partant de `eps = dtype(1)`, divise `eps` par 2 tant que `1 + eps / 2` est différent de 1, puis renvoie `eps`, converti en `float` de Python. Fais **chaque calcul dans le type demandé** : `one = dtype(1)`, `two = dtype(2)`, et `dtype(...)` autour de chaque résultat, pour qu'aucune opération ne passe en `float64` sans que tu le voies. `dtype` vaut `float`, `np.float32` ou `np.float16`.

`np.spacing(x)` donne l'écart entre $x$ et le flottant suivant.

La vérification calcule :
- a) `machine_epsilon() == 2 ** -52` ;
- b) `math.log2(machine_epsilon(np.float32))` (1 décimale) ;
- c) `machine_epsilon(np.float16)` (10 décimales) ;
- d) le nom du type de `machine_epsilon(np.float32)`, en texte ;
- e) `1 + machine_epsilon() / 2 == 1` ;
- f) `math.log2(np.spacing(1000.0) / machine_epsilon())` (1 décimale).

Dans ta copie : combien de fois la boucle tourne-t-elle pour chaque type ? Explique e) : où tombe $1 + \frac{\varepsilon}{2}$, exactement à mi-chemin entre deux flottants, et vers lequel est-il arrondi ? Explique f) avec la puissance de 2 juste en dessous de 1 000.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05a")  # reload your saved file

In [ ]:
with checked("M05a.21a"):
    wbm.check("M05a.21a", mymath.approx.machine_epsilon() == 2 ** -52, computed=True)
with checked("M05a.21b"):
    wbm.check("M05a.21b", math.log2(mymath.approx.machine_epsilon(np.float32)), computed=True)
with checked("M05a.21c"):
    wbm.check("M05a.21c", mymath.approx.machine_epsilon(np.float16), computed=True)
with checked("M05a.21d"):
    wbm.check("M05a.21d", type(mymath.approx.machine_epsilon(np.float32)).__name__, computed=True)
with checked("M05a.21e"):
    wbm.check("M05a.21e", 1 + mymath.approx.machine_epsilon() / 2 == 1, computed=True)
with checked("M05a.21f"):
    wbm.check("M05a.21f", math.log2(np.spacing(1000.0) / mymath.approx.machine_epsilon()), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05a_approx.py", "test_machine_epsilon_")

### Ex M05a.22 — Un log-sum-exp stable · `logsumexp` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer un log-sum-exp qui ne déborde jamais, cas limites compris.  
**Prérequis :** fiche §8 ; Ex M05a.10 · **mymath :** `approx.py` · **Parcours :** C, E

Écris `logsumexp(values)` (fiche §8) : $\ln(e^{v_1} + \cdots + e^{v_n}) = m + \ln\sum_i e^{v_i - m}$, avec $m$ le plus grand des $v_i$. Deux cas à part : une liste vide (une `ValueError`), et une liste dont toutes les valeurs valent `-math.inf` (renvoie `-math.inf` sans calculer $-\infty - (-\infty)$, qui donne `nan`). `math.fsum` additionne une liste de flottants avec un seul arrondi, à la fin.

La vérification calcule :
- a) `logsumexp([2, -1, 0.5])` (6 décimales) ;
- b) `logsumexp([1200, 1199])` (6 décimales) ;
- c) `logsumexp([-1500, -1502])` (6 décimales) ;
- d) `logsumexp([-math.inf, 0.0])` (4 décimales) ;
- e) `logsumexp([-math.inf, -math.inf]) == -math.inf` ;
- f) le nom de l'erreur levée par `logsumexp([])` ;
- g) le nom du type de `logsumexp(np.array([1.0, 2.0]))`, en texte ;
- h) `logsumexp([5.0] * 1000)` (6 décimales).

Dans ta copie : que donnerait le calcul direct `math.log(sum(math.exp(v) for v in values))` pour b) et pour c) ? Vérifie b) et h) avec l'encadrement $m \le \text{LSE} \le m + \ln n$.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M05a")  # reload your saved file

In [ ]:
with checked("M05a.22a"):
    wbm.check("M05a.22a", mymath.approx.logsumexp([2, -1, 0.5]), computed=True)
with checked("M05a.22b"):
    wbm.check("M05a.22b", mymath.approx.logsumexp([1200, 1199]), computed=True)
with checked("M05a.22c"):
    wbm.check("M05a.22c", mymath.approx.logsumexp([-1500, -1502]), computed=True)
with checked("M05a.22d"):
    wbm.check("M05a.22d", mymath.approx.logsumexp([-math.inf, 0.0]), computed=True)
with checked("M05a.22e"):
    wbm.check("M05a.22e", mymath.approx.logsumexp([-math.inf, -math.inf]) == -math.inf, computed=True)
with checked("M05a.22f"):
    wbm.check("M05a.22f", raised(mymath.approx.logsumexp, []), computed=True)
with checked("M05a.22g"):
    wbm.check("M05a.22g", type(mymath.approx.logsumexp(np.array([1.0, 2.0]))).__name__, computed=True)
with checked("M05a.22h"):
    wbm.check("M05a.22h", mymath.approx.logsumexp([5.0] * 1000), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m05a_approx.py", "test_logsumexp_")

### Ex M05a.23 — SymPy `series` et `numpy.finfo` contre tes fonctions 📦 ★ ⏱️ 15 min
**Objectif :** obtenir des développements de Taylor avec SymPy et les limites des flottants avec `numpy.finfo`, et les comparer à tes fonctions.  
**Prérequis :** fiche §3 et §6 · **Parcours :** C

Deux outils tout faits :
- `sp.series(expr, x, a, n)` donne le développement de Taylor de `expr` en `a`, jusqu'au terme de degré $n - 1$, suivi d'un $O(\ldots)$ ; `.removeO()` l'enlève, et `.coeff(x, k)` lit le coefficient de $x^k$. `sp.diff(expr, x, k)` donne la dérivée $k$-ième, et `.subs(x, 0)` sa valeur en 0. Commence par `x = sp.Symbol("x")`.
- `np.finfo(dtype)` décrit un type de flottants : `.eps` (l'epsilon), `.max` (le plus grand), `.tiny` (le plus petit nombre positif **normal**). `np.nextafter(a, b)` donne le flottant qui suit `a` en allant vers `b`.

Calcule :
- a) le coefficient de $x^5$ dans le développement de $\tan x$ en 0 (une fraction irréductible, en texte) ;
- b) le coefficient de $x^3$ dans le développement de $\sqrt{1 + x}$ en 0 (une fraction irréductible, en texte) ;
- c) le plus grand `float16` (un entier) ;
- d) $\log_{10}$ du plus petit `float64` positif normal (2 décimales) ;
- e) `np.nextafter(1.0, 2.0) - 1.0 == np.finfo(float).eps` (`True` ou `False`) ;
- f) la valeur en $x = 0{,}3$ du polynôme de Taylor d'ordre 4 de $e^{\sin x}$ en 0, avec ses dérivées en 0 calculées par SymPy et ta fonction `taylor_polynomial` (5 décimales).

Dans ta copie : comment SymPy affiche-t-il le développement de $\tan x$ ? Que valent les dérivées de f) ? Compare f) à $e^{\sin 0{,}3}$. Pourquoi le plus grand `float16` est-il si petit ? Qu'y a-t-il entre 0 et `tiny` ?

In [ ]:
answer_M05a_23a = ...  # a) an irreducible fraction, as text
answer_M05a_23b = ...  # b) an irreducible fraction, as text
answer_M05a_23c = ...  # c) an integer
answer_M05a_23d = ...  # d) a number (2 decimals)
answer_M05a_23e = ...  # e) True or False
answer_M05a_23f = ...  # f) a number (5 decimals)

In [ ]:
for letter, answer in zip("abcdef", [answer_M05a_23a, answer_M05a_23b, answer_M05a_23c, answer_M05a_23d,
                                     answer_M05a_23e, answer_M05a_23f]):
    wbm.check(f"M05a.23{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §3 et §4. Newton depuis de mauvais points de départ, le domaine de convergence d'une série de Taylor, une fonction et ses polynômes de Taylor.

### Ex M05a.24 — Prédire : Newton depuis un mauvais point de départ 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir le comportement de la méthode de Newton selon la fonction et le point de départ, puis le vérifier.  
**Prérequis :** fiche §4 · **Parcours :** C

On applique la méthode de Newton (sans critère d'arrêt, quelques pas) à cinq cas. Pour a), b) et c), choisis : (1) la suite converge vers une racine ; (2) elle s'éloigne de plus en plus ; (3) elle alterne entre deux valeurs, sans fin ; (4) elle s'arrête : la tangente est horizontale.
- a) $f(x) = \arctan x$ (sa seule racine est 0, et $f'(x) = \frac{1}{1 + x^2}$), depuis $x_0 = 1$ ;
- b) la même fonction, depuis $x_0 = 1{,}5$ ;
- c) $f(x) = x^3 - 2x + 2$, depuis $x_0 = 0$.

Pour d) et e), choisis : (1) la suite converge vers 0, le nombre de décimales justes doublant à chaque pas ; (2) elle converge vers 0, l'erreur divisée par 2 à chaque pas ; (3) elle s'éloigne de 0, la distance doublant à chaque pas ; (4) elle alterne entre 1 et $-1$.
- d) $f(x) = \sqrt[3]{x}$, la racine cubique (négative pour $x < 0$), depuis $x_0 = 1$ ;
- e) $f(x) = x^2$ (une racine double en 0), depuis $x_0 = 1$.

**Avant d'exécuter quoi que ce soit**, écris tes cinq prédictions (le **numéro du choix**), vérifie-les, puis lance l'expérience.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M05a_24a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M05a_24b = ...  # b) 1, 2, 3 or 4
prediction_M05a_24c = ...  # c) 1, 2, 3 or 4
prediction_M05a_24d = ...  # d) 1, 2, 3 or 4
prediction_M05a_24e = ...  # e) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcde", [prediction_M05a_24a, prediction_M05a_24b, prediction_M05a_24c,
                                       prediction_M05a_24d, prediction_M05a_24e]):
    wbm.check(f"M05a.24{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes cinq prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M05a_24a, prediction_M05a_24b, prediction_M05a_24c, prediction_M05a_24d, prediction_M05a_24e]):
    print("⏳ Écris d'abord tes cinq prédictions, puis exécute à nouveau.")
else:
    def iterates(f, fprime, x0, steps):
        """The first values of Newton's sequence from x0 (it stops early on a horizontal tangent)."""
        values = [x0]
        for _ in range(steps):
            x = values[-1]
            slope = fprime(x)
            if slope == 0:
                break
            values.append(x - f(x) / slope)
        return values


    def signed_cube_root(x):
        return math.copysign(abs(x) ** (1 / 3), x)


    def arctan_slope(x):
        return 1 / (1 + x * x)


    print("a) arctan from 1:      ", [round(v, 5) for v in iterates(math.atan, arctan_slope, 1.0, 6)])
    print("b) arctan from 1.5:    ", [round(v, 2) for v in iterates(math.atan, arctan_slope, 1.5, 5)])
    print("c) x³ − 2x + 2 from 0: ", iterates(lambda x: x ** 3 - 2 * x + 2, lambda x: 3 * x ** 2 - 2, 0.0, 6))
    print("d) cube root from 1:   ", [round(v, 3) for v in iterates(signed_cube_root, lambda x: abs(x) ** (-2 / 3) / 3,
                                                                   1.0, 6)])
    print("e) x² from 1:          ", iterates(lambda x: x * x, lambda x: 2 * x, 1.0, 8))

**Après l'expérience**, dans ta copie : pour a) et b), dessine la courbe de $\arctan$ et deux tangentes : pourquoi la tangente en 1,5 renvoie-t-elle plus loin de l'autre côté ? Pour c), fais les deux pas à la main. Pour d), montre que le pas de Newton vaut $x_{k+1} = -2x_k$. Pour e), montre que $x_{k+1} = \frac{x_k}{2}$ : pourquoi la racine double fait-elle perdre la convergence quadratique (fiche §4) ?

### Ex M05a.25 — Taylor de plus en plus loin : où le polynôme décroche 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer l'erreur du polynôme de Taylor de $\ln(1 + x)$ selon le nombre de termes et le point, et voir où la série cesse de converger.  
**Prérequis :** fiche §3 · **Parcours :** C

La fiche (§3) annonce que $\ln(1 + x) = x - \frac{x^2}{2} + \frac{x^3}{3} - \cdots$ seulement pour $-1 < x \le 1$. Vérifie-le.

Écris `log1p_taylor(x, n)`, qui renvoie le polynôme de Taylor d'ordre $n$ de $\ln(1 + x)$ en 0, évalué en $x$ : $\sum_{k=1}^{n} \frac{(-1)^{k+1} x^k}{k}$ ($n$ termes, en `float`).

La vérification compare ta fonction à `math.log1p(x)`, qui calcule $\ln(1 + x)$, et calcule :
- a) l'erreur (la valeur absolue de l'écart) pour $x = 0{,}5$ et $n = 5$ (6 décimales) ;
- b) l'erreur pour $x = 0{,}5$ et $n = 20$, multipliée par $10^8$ (2 décimales) ;
- c) l'erreur pour $x = 1{,}5$ et $n = 5$ (4 décimales) ;
- d) l'erreur pour $x = 1{,}5$ et $n = 20$ (1 décimale) ;
- e) le plus petit $n$ pour lequel l'erreur en $x = 0{,}9$ passe sous $10^{-3}$ (un entier) ;
- f) l'erreur pour $x = 1$ et $n = 100$ (6 décimales).

Une dernière cellule trace l'erreur en fonction de $n$ pour quatre points, sur une échelle logarithmique.

Dans ta copie : comment l'erreur évolue-t-elle avec $n$ pour chaque point ? Pour $|x| < 1$, compare l'erreur au premier terme négligé, $\frac{|x|^{n+1}}{n+1}$. Pourquoi la série ne peut-elle pas converger pour $x = 1{,}5$ (regarde la taille du terme $\frac{x^k}{k}$) ? Combien de termes faudrait-il, en $x = 1$, pour une erreur de $10^{-6}$ ?

In [ ]:
def log1p_taylor(x, n):
    """The Taylor polynomial of order n of ln(1 + x) at 0, evaluated at x: the sum of (-1)**(k+1) x**k / k for k
    from 1 to n."""
    raise NotImplementedError("une boucle sur k de 1 à n, ou sum(...)")

In [ ]:
with checked("M05a.25a"):
    wbm.check("M05a.25a", abs(log1p_taylor(0.5, 5) - math.log1p(0.5)), computed=True)
with checked("M05a.25b"):
    wbm.check("M05a.25b", abs(log1p_taylor(0.5, 20) - math.log1p(0.5)) * 1e8, computed=True)
with checked("M05a.25c"):
    wbm.check("M05a.25c", abs(log1p_taylor(1.5, 5) - math.log1p(1.5)), computed=True)
with checked("M05a.25d"):
    wbm.check("M05a.25d", abs(log1p_taylor(1.5, 20) - math.log1p(1.5)), computed=True)
with checked("M05a.25e"):
    wbm.check("M05a.25e", next((n for n in range(1, 1001) if abs(log1p_taylor(0.9, n) - math.log1p(0.9)) < 1e-3), -1),
              computed=True)
with checked("M05a.25f"):
    wbm.check("M05a.25f", abs(log1p_taylor(1.0, 100) - math.log(2)), computed=True)

**La figure** (elle utilise ta fonction).

In [ ]:
with checked("M05a.25 figure"):
    ns_25 = range(1, 41)
    errors_25 = {x0: [abs(log1p_taylor(x0, n) - math.log1p(x0)) for n in ns_25] for x0 in (0.5, 0.9, 1.0, 1.5)}
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for x0, errors in errors_25.items():
        ax.semilogy(ns_25, errors, marker=".", label=f"x = {x0}")
    ax.set(xlabel="n (nombre de termes)", ylabel="erreur", title="ln(1 + x) et son polynôme de Taylor d'ordre n")
    ax.legend()
    plt.show()

### Ex M05a.26 — Une fonction et ses polynômes de Taylor d'ordre 1, 2 et 3 🎨 ★ ⏱️ 20 min
**Objectif :** tracer une fonction et ses polynômes de Taylor en un point, et voir l'approximation s'améliorer près du point et se dégrader loin de lui.  
**Prérequis :** fiche §3 ; Ex M05a.18 ; R0 (matplotlib) · **Parcours :** C, E

Écris `draw_taylor(ax, f, derivatives, a, orders)`, qui dessine dans l'axe `ax`, sur une grille d'au moins 200 points de $[a - 2, a + 2]$ :
- la fonction `f` (une fonction qui accepte un array, comme `np.sin`) ;
- pour chaque ordre $n$ de `orders`, le polynôme de Taylor d'ordre $n$ en `a`, construit avec les $n + 1$ premières valeurs de `derivatives` ($[f(a), f'(a), \ldots]$) ;
- le point de contact $(a, f(a))$, marqué par un point ;

chaque courbe avec un label, puis la légende. Ta fonction `mymath.approx.taylor_polynomial` donne la valeur d'un polynôme en un point (une compréhension de liste la calcule sur toute la grille), ou bien une somme de termes sur un array.

Puis trace $\sin$ et ses polynômes d'ordre 1, 2 et 3 en $a = \frac{\pi}{4}$, dont les dérivées valent $\frac{\sqrt{2}}{2}, \frac{\sqrt{2}}{2}, -\frac{\sqrt{2}}{2}, -\frac{\sqrt{2}}{2}$.

Dans ta copie : sur quelle largeur autour de $\frac{\pi}{4}$ chaque polynôme reste-t-il proche de la courbe, à l'œil ? Le polynôme d'ordre 2 est-il au-dessus ou au-dessous de la courbe, et pourquoi (signe de $f''$, §1) ? Pourquoi chaque ordre de plus améliore-t-il l'approximation près de $a$ et pas forcément loin de $a$ ?

In [ ]:
def draw_taylor(ax, f, derivatives, a, orders):
    """In ax, on [a - 2, a + 2]: f, its Taylor polynomials at a of the given orders (from derivatives = [f(a), f'(a),
    ...]) and the point (a, f(a)), each with a label; then the legend."""
    raise NotImplementedError("une grille, ax.plot pour f puis pour chaque ordre, le point (a, f(a)), ax.legend()")

In [ ]:
with checked("M05a.26 figure"):
    fig, ax = plt.subplots(figsize=(7, 4))
    root_half = math.sqrt(2) / 2
    try:
        created = drawn(draw_taylor, ax, np.sin, [root_half, root_half, -root_half, -root_half], math.pi / 4,
                        (1, 2, 3))
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M05a.26 : draw_taylor ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans l'axe reçu.")
    else:
        plt.show()

**Vérifications automatiques** (sur un autre exemple ; la figure elle-même se juge à l'œil).

In [ ]:
with checked("M05a.26 vérifications"):
    problems = []
    a_t = 0.5                                                          # another example: exp at 0.5, orders 1 and 3
    ders_t = [math.exp(a_t)] * 4

    def poly_t(order, xs):
        return sum(ders_t[k] * (xs - a_t) ** k / math.factorial(k) for k in range(order + 1))

    expected = {"la fonction": np.exp, "le polynôme d'ordre 1": lambda xs: poly_t(1, xs),
                "le polynôme d'ordre 3": lambda xs: poly_t(3, xs)}
    fig, ax = plt.subplots(figsize=(6, 4))
    try:
        created = drawn(draw_taylor, ax, np.exp, ders_t, a_t, (1, 3))
    finally:
        plt.close(fig)                                                 # only the checks below are shown here
    if created:
        problems.append("draw_taylor ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        curves = []
        for line in ax.get_lines():
            xs = np.asarray(line.get_xdata(), dtype=float)
            ys = np.asarray(line.get_ydata(), dtype=float)
            if xs.size >= 50:
                curves.append((line, xs, ys))
        for name, function in expected.items():
            match = [line for line, xs, ys in curves if np.allclose(ys, function(xs), rtol=1e-6, atol=1e-9)]
            if not match:
                problems.append(f"aucune courbe d'au moins 50 points n'est {name}" + (
                    "" if name == "la fonction" else " (le terme d'ordre k est f⁽ᵏ⁾(a) (x − a)^k / k!, pour k de 0 "
                                                     "à l'ordre)"))
            elif not match[0].get_label() or match[0].get_label().startswith("_"):
                problems.append(f"{name} : donne-lui un label, pour la légende")
        if curves and max(xs.max() - xs.min() for _, xs, _ in curves) < 3.9:
            problems.append("trace les courbes sur tout l'intervalle [a − 2, a + 2]")
        marked = any(np.any(np.isclose(np.asarray(line.get_xdata(), dtype=float), a_t))
                     and np.any(np.isclose(np.asarray(line.get_ydata(), dtype=float), math.exp(a_t)))
                     and len(line.get_xdata()) < 5 for line in ax.get_lines())
        marked = marked or any(np.any(np.all(np.isclose(c.get_offsets(), [a_t, math.exp(a_t)]), axis=1))
                               for c in ax.collections if len(c.get_offsets()))
        if not marked:
            problems.append("marque le point (a, f(a)) par un point, par exemple ax.plot(a, f(a), \"o\")")
        if ax.get_legend() is None:
            problems.append("il manque la légende (ax.legend())")
    if problems:
        print("❌ Ex M05a.26 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M05a.26 : la fonction, ses polynômes de Taylor et le point de contact sont justes, avec une "
              "légende.")

## Partie C · Erreurs classiques, savoir-faire et défi

Fiche §4, §6 et §8. Quatre erreurs d'une softmax à trouver, la tolérance d'un test de flottants, puis une racine cubique en quelques pas.

### Ex M05a.27 — Chasse à l'erreur : une softmax qui renvoie `nan` 🐛 ★ ⏱️ 15 min
**Objectif :** repérer dans du code les erreurs classiques d'une softmax et d'un log-sum-exp, de formule ou de calcul en flottants, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §6 et §8 ; Ex M05a.10 · **Parcours :** C

Un modèle donne trois scores $\mathbf{v} = (1000,\ 999,\ 200)$. Une collègue a écrit six lignes, chacune avec un commentaire qui dit ce qu'elle veut calculer ; chaque ligne utilise les variables des lignes précédentes telles qu'elles **devraient** être (on suppose les lignes précédentes justes). Quatre lignes ne calculent pas ce que dit leur commentaire : une erreur de formule, ou un calcul qui échoue en `float64`. **Sans exécuter de code**, trouve-les.

```python
v = [1000.0, 999.0, 200.0]
m = min(v)                                        # (A) the largest score
total = sum(math.exp(x - m) for x in v)           # (B) Σ exp(v_i − m), with m the largest score
lse = math.log(total)                             # (C) log-sum-exp of v
p1 = math.exp(v[0] - m) / total                   # (D) softmax(v)[0]
log_p3 = math.log(math.exp(v[2] - m) / total)     # (E) log-softmax of the third score
s = sum(math.exp(x - m) for x in v) / len(v)      # (F) the sum of the three probabilities of softmax(v)
```

Questions :
- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des six lignes, de (A) à (F), la bonne valeur de ce que dit son commentaire (une liste de 6 nombres, 3 décimales).

Dans ta copie : pour chaque ligne fausse, quelle erreur a été faite, et comment la corriger ? Laquelle donne une erreur à l'exécution plutôt qu'une valeur fausse ? D'où vient le `nan` du titre, quand la ligne (A) est fausse (calcule `math.exp(1000 - 200)` en NumPy, puis `inf / inf`) ?

In [ ]:
answer_M05a_27a = ...  # a) letters, as text
answer_M05a_27b = ...  # b) a list of 6 values (3 decimals)

In [ ]:
wbm.check("M05a.27a", answer_M05a_27a)
wbm.check("M05a.27b", answer_M05a_27b)

### Ex M05a.28 — Erreur relative ou absolue : choisir la tolérance d'un test 🛠️ ★ ⏱️ 10 min
**Objectif :** écrire la comparaison de deux flottants qu'utilisent les tests, avec une tolérance relative et une tolérance absolue, et savoir laquelle choisir.  
**Prérequis :** fiche §6 · **Parcours :** C

On ne compare jamais deux flottants calculés avec `==` : `0.1 + 0.2 == 0.3` est faux (R0). On vérifie qu'ils sont **proches**. Mais proches comment ?
- Une tolérance **absolue**, $|a - b| \le 10^{-9}$, est absurde pour des nombres de l'ordre de $10^{20}$, où deux flottants voisins sont déjà distants de plus de 10 000 (§6).
- Une tolérance **relative**, $|a - b| \le 10^{-9}\,\max(|a|, |b|)$, est absurde près de 0 : un résultat qui devrait valoir 0 et vaut $10^{-17}$ (une annulation, §6) n'est « proche » d'aucun nombre.

Les bibliothèques combinent donc les deux. Écris `close_enough(got, expected, rel_tol=1e-9, abs_tol=0.0)`, qui renvoie `True` si et seulement si

$$|\text{got} - \text{expected}| \le \max\big(\text{rel\_tol} \times \max(|\text{got}|, |\text{expected}|),\ \text{abs\_tol}\big),$$

avec trois règles : deux valeurs égales (même infinies) sont proches ; `nan` n'est proche de rien, pas même de lui-même ; une infinie n'est proche que d'elle-même. Une tolérance négative lève une `ValueError`. C'est la définition de `math.isclose`, qui est donc interdite ici : la vérification compare tes réponses aux siennes, sur des cas choisis.

Dans ta copie : quelle tolérance choisirais-tu pour tester que `logsumexp([1000, 1000])` vaut $1000 + \ln 2$ ? Pour tester qu'une erreur d'arrondi, qui devrait être nulle, est négligeable ? `numpy.isclose(a, b)` utilise $|a - b| \le \text{atol} + \text{rtol}\,|b|$, avec par défaut `atol=1e-8` : pourquoi n'est-elle pas symétrique, et quel piège cache son `atol` pour de très petits nombres ?

In [ ]:
def close_enough(got, expected, rel_tol=1e-9, abs_tol=0.0):
    """True when |got - expected| <= max(rel_tol * max(|got|, |expected|), abs_tol), like math.isclose (without
    using it). ValueError for a negative tolerance."""
    raise NotImplementedError("les tolérances, puis l'égalité, nan et les infinis, puis l'inégalité")

In [ ]:
with checked("M05a.28 vérifications"):
    problems = []
    cases = [((1.0, 1.0 + 1e-10), {}), ((1e20, 1e20 + 1e5), {}), ((1e-20, 2e-20), {}), ((0.0, 1e-12), {}),
             ((0.0, 1e-12), {"abs_tol": 1e-9}), ((100.0, 101.0), {"rel_tol": 0.01}),
             ((100.0, 102.0), {"rel_tol": 0.01}), ((1.0, 1.1), {"rel_tol": 0.091}), ((1.1, 1.0), {"rel_tol": 0.091}),
             ((5.0, 5.0), {"rel_tol": 0.0}), ((math.inf, math.inf), {}), ((math.inf, 1e308), {"rel_tol": 0.5}),
             ((math.nan, math.nan), {}), ((-3.0, 3.0), {"rel_tol": 2.5})]
    for (got, expected), options in cases:
        reference = math.isclose(got, expected, **options)
        call = f"close_enough({got!r}, {expected!r}" + "".join(f", {k}={v!r}" for k, v in options.items()) + ")"
        try:
            answer = close_enough(got, expected, **options)
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"{call} lève {type(exc).__name__} ({exc}) ; attendu {reference}")
            continue
        if answer is not reference:
            problems.append(f"{call} : attendu {reference}, obtenu {answer!r}")
    for options in ({"rel_tol": -1e-9}, {"abs_tol": -1.0}):
        try:
            close_enough(1.0, 1.0, **options)
        except ValueError:
            pass
        except NotImplementedError:
            raise
        except Exception as exc:  # noqa: BLE001
            problems.append(f"une tolérance négative ({options}) lève {type(exc).__name__} : attendu une ValueError")
        else:
            problems.append(f"une tolérance négative ({options}) est acceptée : attendu une ValueError")
    if problems:
        print("❌ Ex M05a.28 : à revoir : " + " ; ".join(problems[:5]) + ("…" if len(problems) > 5 else "."))
    else:
        print("✅ Ex M05a.28 : close_enough donne les mêmes réponses que math.isclose, cas limites compris.")

### Ex M05a.29 — Défi : une racine cubique à 10⁻¹² en le moins d'itérations possible 🏆 ★★★ ⏱️ 25 min
**Objectif :** programmer une racine cubique par la méthode de Newton, avec un point de départ assez bon pour converger en quelques pas, quelle que soit la taille du nombre.  
**Prérequis :** fiche §4 et §6 ; Ex M05a.12 et M05a.19 · **Parcours :** C

Écris `cube_root(a)`, qui renvoie le couple `(x, steps)` : une approximation de $\sqrt[3]{a}$ pour $a > 0$, obtenue par la méthode de Newton sur $f(x) = x^3 - a$, et le nombre de pas de Newton effectués. Un pas s'écrit $x \leftarrow x - \frac{x^3 - a}{3x^2} = \frac{1}{3}\big(2x + \frac{a}{x^2}\big)$. Arrête-toi quand deux valeurs successives sont presque égales (par exemple $|x_{k+1} - x_k| \le 10^{-15}\,x_{k+1}$).

**L'objectif** : une erreur relative inférieure à $10^{-12}$ en **6 pas au plus**, pour des nombres de $10^{-300}$ à $10^{300}$. Tout se joue au point de départ : la convergence quadratique (M05a.12) n'opère que près de la racine, et partir de $x_0 = a$ ou de $x_0 = 1$ coûte plus de cent pas pour $a = 10^{30}$. Une piste : `m, e = math.frexp(a)` écrit $a = m \times 2^e$ avec $0{,}5 \le m < 1$ ; avec $e = 3q + r$ (`divmod(e, 3)`), $\sqrt[3]{a} = \sqrt[3]{m \times 2^r} \times 2^q$, et il ne reste qu'à approcher $\sqrt[3]{y}$ pour $y = m \times 2^r$ entre 0,5 et 4, par exemple par une droite.

La vérification calcule ta racine pour onze nombres, de $5 \times 10^{-300}$ à $10^{300}$, affiche la pire erreur relative et le pire nombre de pas, puis vérifie `cube_root(50.0)[0]` (10 décimales).

Dans ta copie : l'erreur relative de ton point de départ, au pire ? Combien de pas faut-il alors, d'après $\varepsilon_{k+1} \approx \varepsilon_k^2$ (pour $x^3 - a$, l'erreur relative est à peu près élevée au carré à chaque pas) ? Pourquoi, depuis $x_0 = 1$ et pour $a = 10^{30}$, la suite ne fait-elle que se multiplier par environ $\frac{2}{3}$ pendant des dizaines de pas ?

In [ ]:
def cube_root(a):
    """The cube root of a > 0 by Newton's method on x**3 - a, and the number of Newton steps taken: (x, steps)."""
    raise NotImplementedError("un bon point de départ (math.frexp), puis des pas de Newton jusqu'à l'arrêt")

In [ ]:
CASES_29 = [50.0, 2.0, 0.008, 1e-30, 3.7e25, 123456.789, 1e300, 5e-300, 0.9999, 7.0e-5, 1.0]
with checked("M05a.29 défi"):
    worst_error, worst_steps = 0.0, 0
    for a in CASES_29:
        root, steps = cube_root(a)
        exact = a ** (1 / 3)
        worst_error = max(worst_error, abs(root - exact) / exact)
        worst_steps = max(worst_steps, steps)
    print(f"Pire erreur relative : {worst_error:.1e} · pire nombre d'itérations : {worst_steps}")
    if not worst_error < 1e-12:
        print("❌ Ex M05a.29 : la précision n'est pas atteinte (objectif : erreur relative < 1e-12) : continue les pas "
              "de Newton jusqu'à ce que deux valeurs successives soient presque égales.")
    elif worst_steps > 6:
        print("❌ Ex M05a.29 : précision atteinte, mais trop d'itérations (objectif : 6 au plus) : soigne le point de "
              "départ (math.frexp).")
    else:
        print("✅ Ex M05a.29 : défi réussi, une précision de 1e-12 en 6 itérations au plus, de 1e-300 à 1e300.")
with checked("M05a.29"):
    wbm.check("M05a.29", cube_root(50.0)[0], computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu étudier la convexité d'une fonction et trouver ses extrema globaux sur un intervalle ?
2. Sais-tu écrire un polynôme de Taylor, borner son erreur, et dire à quel ordre une différence finie approche une dérivée ?
3. Sais-tu utiliser la méthode de Newton, et dire quand elle converge vite ?
4. Sais-tu ce qu'un `float64` représente exactement, et calculer un log-sum-exp ou une softmax sans overflow ?

**La suite** : le chapitre 5 de workbookIA (dérivées, différences finies, gradient), puis M05b, les fonctions de plusieurs variables et le gradient.